In [ ]:
# -*- coding: utf-8 -*-
"""
Import des donnees socio-demographiques vers SQL Server (staging).

Deux sources :
  1. OPENDATA_SECTOREN_2025 (Statbel) -> staging_secteur_stat
     Population et superficie par secteur statistique, filtre sur Bruxelles.
  2. Statistique fiscale IBSA, feuille 2.1.5.1 -> staging_revenu_commune
     Revenu equivalent median par commune, depivote en format long.

Usage :
    python import_socio.py --dry-run     # lit et affiche, sans toucher a la base
    python import_socio.py               # lit et insere

Le script est idempotent : chaque table est videe avant reinsertion.
"""

import sys
import os
import openpyxl
import pyodbc

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------

SERVEUR = "ICT-202-11"
BASE = "TFE_STIB"

DOSSIER = r"C:\Users\stgadmin\Desktop\TFE-STIB"

FICHIER_SECTEURS = os.path.join(DOSSIER, "OPENDATA_SECTOREN_2025_NEW (1).xlsx")
FICHIER_REVENUS = os.path.join(
    DOSSIER, "2.1b_rev_dep_menages_stat_fisc_menages_prives_20260526.xlsx"
)

FEUILLE_REVENUS = "2.1.5.1"   # revenu equivalent MEDIAN par habitant, par commune

# Les 19 communes bruxelloises avec leur code INS (CD_REFNIS).
# Sert a la fois de filtre et de cle de jointure entre les deux fichiers.
COMMUNES_BXL = {
    "Anderlecht": "21001",
    "Auderghem": "21002",
    "Berchem-Sainte-Agathe": "21003",
    "Bruxelles": "21004",
    "Etterbeek": "21005",
    "Evere": "21006",
    "Forest": "21007",
    "Ganshoren": "21008",
    "Ixelles": "21009",
    "Jette": "21010",
    "Koekelberg": "21011",
    "Molenbeek-Saint-Jean": "21012",
    "Saint-Gilles": "21013",
    "Saint-Josse-ten-Noode": "21014",
    "Schaerbeek": "21015",
    "Uccle": "21016",
    "Watermael-Boitsfort": "21017",
    "Woluwe-Saint-Lambert": "21018",
    "Woluwe-Saint-Pierre": "21019",
}

PREFIXE_REFNIS_BXL = "21"     # tous les codes INS bruxellois commencent par 21


# ----------------------------------------------------------------------
# 1. LECTURE DU FICHIER STATBEL (secteurs statistiques)
# ----------------------------------------------------------------------

def lire_secteurs(chemin):
    """
    Table plate : en-tetes en ligne 1, donnees a partir de la ligne 2.
    On ne garde que les lignes dont CD_REFNIS commence par 21 (Bruxelles).
    """
    wb = openpyxl.load_workbook(chemin, read_only=True, data_only=True)
    ws = wb["Blad1"]

    lignes = []
    total_lu = 0

    for r in ws.iter_rows(min_row=2, values_only=True):
        if r[0] is None:
            continue
        total_lu += 1

        cd_refnis = str(r[0]).strip()
        if not cd_refnis.startswith(PREFIXE_REFNIS_BXL):
            continue

        lignes.append((
            cd_refnis,
            str(r[1]).strip() if r[1] is not None else None,   # CD_SECTOR
            str(r[2]) if r[2] is not None else None,           # TOTAL (population)
            str(r[5]) if r[5] is not None else None,           # superficie hm2
            str(r[7]).strip() if r[7] is not None else None,   # nom secteur FR
            str(r[9]).strip() if r[9] is not None else None,   # nom commune FR
        ))

    wb.close()
    print(f"[secteurs] lignes lues : {total_lu} | retenues (Bruxelles) : {len(lignes)}")
    return lignes


# ----------------------------------------------------------------------
# 2. LECTURE DU FICHIER IBSA (revenus) + DEPIVOTEMENT
# ----------------------------------------------------------------------

def lire_revenus(chemin, feuille):
    """
    Fichier de publication : titre lignes 1-3, en-tetes (annees) ligne 4,
    communes ET agregats melanges ensuite, notes de bas de tableau a la fin.

    Strategie : on ne se fie pas aux numeros de ligne. On parcourt tout et on
    ne garde que les lignes dont le libelle figure dans COMMUNES_BXL.
    Les agregats (Region, Belgique...) et les notes sont ainsi ecartes
    automatiquement.

    Le tableau est en format LARGE (une colonne par annee). On le depivote
    en format LONG : une ligne = une commune x une annee.
    """
    wb = openpyxl.load_workbook(chemin, read_only=True, data_only=True)
    ws = wb[feuille]

    grille = list(ws.iter_rows(values_only=True))
    wb.close()

    # --- reperage de la ligne d'en-tetes : celle qui contient des annees ---
    idx_entete = None
    for i, r in enumerate(grille):
        annees_trouvees = [
            c for c in r[1:]
            if c is not None and str(c).strip().isdigit() and len(str(c).strip()) == 4
        ]
        if len(annees_trouvees) >= 3:
            idx_entete = i
            break

    if idx_entete is None:
        raise RuntimeError("Ligne d'en-tetes (annees) introuvable.")

    entete = grille[idx_entete]
    # position -> annee, uniquement pour les colonnes qui portent une annee
    colonnes_annees = {
        j: str(c).strip()
        for j, c in enumerate(entete)
        if j > 0 and c is not None
        and str(c).strip().isdigit() and len(str(c).strip()) == 4
    }
    print(f"[revenus] en-tetes ligne {idx_entete + 1} | "
          f"annees detectees : {len(colonnes_annees)} "
          f"({min(colonnes_annees.values())}-{max(colonnes_annees.values())})")

    # --- depivotement ---
    lignes = []
    communes_vues = set()

    for r in grille[idx_entete + 1:]:
        if not r or r[0] is None:
            continue
        libelle = str(r[0]).strip()
        if libelle not in COMMUNES_BXL:
            continue                      # ecarte agregats et notes
        communes_vues.add(libelle)

        for j, annee in colonnes_annees.items():
            valeur = r[j] if j < len(r) else None
            if valeur is None:
                continue
            lignes.append((
                COMMUNES_BXL[libelle],    # cd_refnis, cle de jointure
                libelle,
                annee,
                str(valeur),
            ))

    manquantes = set(COMMUNES_BXL) - communes_vues
    if manquantes:
        print("[revenus] ATTENTION, communes non trouvees :", sorted(manquantes))

    print(f"[revenus] communes retenues : {len(communes_vues)}/19 | "
          f"lignes depivotees : {len(lignes)}")
    return lignes


# ----------------------------------------------------------------------
# 3. INSERTION EN BASE
# ----------------------------------------------------------------------

def connexion():
    return pyodbc.connect(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={SERVEUR};DATABASE={BASE};Trusted_Connection=yes;"
    )


def charger(nom_table, colonnes, lignes):
    """Vide la table puis insere les lignes. Idempotent."""
    conn = connexion()
    cur = conn.cursor()
    cur.fast_executemany = True

    cur.execute(f"DELETE FROM {nom_table}")

    placeholders = ",".join("?" * len(colonnes))
    sql = f"INSERT INTO {nom_table} ({','.join(colonnes)}) VALUES ({placeholders})"
    cur.executemany(sql, lignes)
    conn.commit()

    cur.execute(f"SELECT COUNT(*) FROM {nom_table}")
    total = cur.fetchone()[0]

    cur.close()
    conn.close()
    print(f"  -> {nom_table} : {total} lignes en base")


# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------

if __name__ == "__main__":
    dry_run = "--dry-run" in sys.argv

    secteurs = lire_secteurs(FICHIER_SECTEURS)
    revenus = lire_revenus(FICHIER_REVENUS, FEUILLE_REVENUS)

    print("\nApercu secteurs :", secteurs[0])
    print("Apercu revenus  :", revenus[0], "...", revenus[-1])

    if dry_run:
        print("\n[dry-run] rien n'a ete ecrit en base.")
        sys.exit(0)

    charger(
        "staging_secteur_stat",
        ["cd_refnis", "cd_sector", "population",
         "superficie_hm2", "nom_secteur_fr", "nom_commune_fr"],
        secteurs,
    )
    charger(
        "staging_revenu_commune",
        ["cd_refnis", "nom_commune", "annee", "revenu_median"],
        revenus,
    )
    print("\nTermine.")